In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# !pip install optuna

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import BertTokenizer, BertForSequenceClassification, AdamW, DistilBertForSequenceClassification
from sklearn.metrics import mean_squared_error, mean_absolute_error, cohen_kappa_score
from scipy.stats import pearsonr, spearmanr
from sklearn.preprocessing import MinMaxScaler
import numpy as np
from transformers import (AutoConfig,
                          AutoModelForSequenceClassification,
                          AutoTokenizer, AdamW,
                          get_linear_schedule_with_warmup,
                          set_seed,
                          )
# import optuna
# from optuna.integration import PyTorchLightningPruningCallback

In [4]:
import torch
import pandas as pd
from torch.utils.data import DataLoader, Dataset
from transformers import RobertaTokenizer, RobertaForSequenceClassification, AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import pearsonr, spearmanr
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, cohen_kappa_score, r2_score
from torch.optim.lr_scheduler import StepLR
from torch.optim.lr_scheduler import ReduceLROnPlateau

In [5]:
!pip install indic-nlp-library

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 811.3 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 11.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.1/121.1 kB 13.9 MB/s eta 0:00:00


In [6]:
df = pd.read_csv("/content/drive/MyDrive/EG/translated_dataset/mbart50/mbart50_p7.csv")

In [7]:
df.head(2)

,Unnamed: 0,essay_id,essay_set,essay,score,essay_hindi,mbart50_m2m
0,10686,17834,7,Patience is when your waiting .I was patience ...,15,सब्र तब होता है जब आप इंतजार कर रहे होते हैं। ...,धैर्य जब तपाईँले पर्खिरहनु भएको हो। म लंचको ला...
1,10687,17836,7,"I am not a patience person, like I can’t sit i...",13,"मैं एक धैर्यवान व्यक्ति नहीं हूं, जैसे कि मैं ...","म चिन्तित व्यक्ति होइन, जस्तो म पाँच मिनेटभन्द..."


In [8]:
import re
Tweet = []
for tweettext in df["mbart50_m2m"]:
  text = re.sub(r"http\S+", "", tweettext)
  text = re.sub(r"@\S+","",text)
  emoji_pattern = re.compile("["
                               u"\U0001F600-\U0001F64F"  # emoticons
                               u"\U0001F300-\U0001F5FF"  # symbols & pictographs
                               u"\U0001F680-\U0001F6FF"  # transport & map symbols
                               u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
                               u"\U00002500-\U00002BEF"  # chinese char
                               u"\U00002702-\U000027B0"
                               u"\U00002702-\U000027B0"
                               u"\U000024C2-\U0001F251"
                               u"\U0001f926-\U0001f937"
                               u"\U00010000-\U0010ffff"
                               u"\u2640-\u2642"
                               u"\u2600-\u2B55"
                               u"\u200d"
                               u"\u23cf"
                               u"\u23e9"
                               u"\u231a"
                               u"\ufe0f"  # dingbats
                               u"\u3030"
                               "]+", flags=re.UNICODE)

  text = emoji_pattern.sub(r'',text)
  text = re.sub('[A-Za-z]+', ' ', text) #Remove english alphabets
  #print(text)
  Tweet.append(text)

In [9]:
df["mbart50_m2m"] = pd.Series(Tweet)
df.head()

,Unnamed: 0,essay_id,essay_set,essay,score,essay_hindi,mbart50_m2m
0,10686,17834,7,Patience is when your waiting .I was patience ...,15,सब्र तब होता है जब आप इंतजार कर रहे होते हैं। ...,धैर्य जब तपाईँले पर्खिरहनु भएको हो। म लंचको ला...
1,10687,17836,7,"I am not a patience person, like I can’t sit i...",13,"मैं एक धैर्यवान व्यक्ति नहीं हूं, जैसे कि मैं ...","म चिन्तित व्यक्ति होइन, जस्तो म पाँच मिनेटभन्द..."
2,10688,17837,7,One day I was at basketball practice and I was...,15,एक दिन मैं बास्केटबॉल अभ्यास में था और मैं अपन...,एक दिन म बास्केटबाल अभ्यासमा थिएँ र मैले आफ्नो...
3,10689,17838,7,I going to write about a time when I went to t...,17,मैं उस समय के बारे में लिखने जा रहा हूं जब मैं...,"फेयरमा जाँदा हामीले रमाइलो गर्यौं, हामीले ह..."
4,10690,17839,7,It can be very hard for somebody to be patient...,13,किसी के लिए धैर्य रखना बहुत कठिन हो सकता है। य...,कसैको लागि धैर्य धेरै कठिन हुन सक्छ । यदि तिमी...


In [10]:

df['mbart50_m2m'] = df['mbart50_m2m'].replace(r'\r+|\n+|\t+','', regex=True)
df['mbart50_m2m'] = df['mbart50_m2m'].str.replace("  ", " ")

In [11]:
# tokenization
from indicnlp.tokenize import indic_tokenize
def tokenization(indic_string):
    tokens = []
    for t in indic_tokenize.trivial_tokenize(indic_string):
        tokens.append(t)
    return tokens
df['mbart50_m2m'] = df['mbart50_m2m'].apply(lambda x: tokenization(x))

In [12]:
df.head(5)

,Unnamed: 0,essay_id,essay_set,essay,score,essay_hindi,mbart50_m2m
0,10686,17834,7,Patience is when your waiting .I was patience ...,15,सब्र तब होता है जब आप इंतजार कर रहे होते हैं। ...,"[धैर्य, जब, तपाईँले, पर्खिरहनु, भएको, हो, ।, म..."
1,10687,17836,7,"I am not a patience person, like I can’t sit i...",13,"मैं एक धैर्यवान व्यक्ति नहीं हूं, जैसे कि मैं ...","[म, चिन्तित, व्यक्ति, होइन, ,, जस्तो, म, पाँच,..."
2,10688,17837,7,One day I was at basketball practice and I was...,15,एक दिन मैं बास्केटबॉल अभ्यास में था और मैं अपन...,"[एक, दिन, म, बास्केटबाल, अभ्यासमा, थिएँ, र, मै..."
3,10689,17838,7,I going to write about a time when I went to t...,17,मैं उस समय के बारे में लिखने जा रहा हूं जब मैं...,"[फेयरमा, जाँदा, हामीले, रमाइलो, गर्यौं, ,, हाम..."
4,10690,17839,7,It can be very hard for somebody to be patient...,13,किसी के लिए धैर्य रखना बहुत कठिन हो सकता है। य...,"[कसैको, लागि, धैर्य, धेरै, कठिन, हुन, सक्छ, ।,..."


In [13]:
# Remove ‘\n’ from each tokenized
for i in range(len(df)):
    df['mbart50_m2m'][i] = [s.replace("\n", "") for s in df['mbart50_m2m'][i]]

<ipython-input-13-a24e87f8c9d4>:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['mbart50_m2m'][i] = [s.replace("\n", "") for s in df['mbart50_m2m'][i]]


In [14]:
# Remove Punctuations

punctuations = ['nn','n','–', '।','/', '`', '+', '\\', '"', '?', '▁(', '$', '@', '[', '_', "\'", '!', ',', ':', '^', '|', ']', '=', '%', '&', '.', ')', '(', "#", '*', '', ';', '-', '}','|','"']


to_be_removed = punctuations

for i in range(len(df)):
    df['mbart50_m2m'][i]=[ele for ele in df['mbart50_m2m'][i] if ele not in (to_be_removed)]
# count_length()
df.head(2)

<ipython-input-14-117b6cc3cbed>:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['mbart50_m2m'][i]=[ele for ele in df['mbart50_m2m'][i] if ele not in (to_be_removed)]


,Unnamed: 0,essay_id,essay_set,essay,score,essay_hindi,mbart50_m2m
0,10686,17834,7,Patience is when your waiting .I was patience ...,15,सब्र तब होता है जब आप इंतजार कर रहे होते हैं। ...,"[धैर्य, जब, तपाईँले, पर्खिरहनु, भएको, हो, म, ल..."
1,10687,17836,7,"I am not a patience person, like I can’t sit i...",13,"मैं एक धैर्यवान व्यक्ति नहीं हूं, जैसे कि मैं ...","[म, चिन्तित, व्यक्ति, होइन, जस्तो, म, पाँच, मि..."


In [15]:
# #Remove comma as a seperators
for i in range(len(df)):
    df['mbart50_m2m'][i] = ' '.join(df['mbart50_m2m'][i])
df.head(5)

<ipython-input-15-01822b43c9e6>:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['mbart50_m2m'][i] = ' '.join(df['mbart50_m2m'][i])


,Unnamed: 0,essay_id,essay_set,essay,score,essay_hindi,mbart50_m2m
0,10686,17834,7,Patience is when your waiting .I was patience ...,15,सब्र तब होता है जब आप इंतजार कर रहे होते हैं। ...,धैर्य जब तपाईँले पर्खिरहनु भएको हो म लंचको लाग...
1,10687,17836,7,"I am not a patience person, like I can’t sit i...",13,"मैं एक धैर्यवान व्यक्ति नहीं हूं, जैसे कि मैं ...",म चिन्तित व्यक्ति होइन जस्तो म पाँच मिनेटभन्दा...
2,10688,17837,7,One day I was at basketball practice and I was...,15,एक दिन मैं बास्केटबॉल अभ्यास में था और मैं अपन...,एक दिन म बास्केटबाल अभ्यासमा थिएँ र मैले आफ्नो...
3,10689,17838,7,I going to write about a time when I went to t...,17,मैं उस समय के बारे में लिखने जा रहा हूं जब मैं...,फेयरमा जाँदा हामीले रमाइलो गर्यौं हामीले हेर्न...
4,10690,17839,7,It can be very hard for somebody to be patient...,13,किसी के लिए धैर्य रखना बहुत कठिन हो सकता है। य...,कसैको लागि धैर्य धेरै कठिन हुन सक्छ यदि तिमी ध...


In [16]:
len(df)

1569

In [17]:
df = df.dropna(subset=['score'])
df.reset_index(drop=True, inplace=True)

In [18]:
print(df.isnull().any())

Unnamed: 0     False
essay_id       False
essay_set      False
essay          False
score          False
essay_hindi    False
mbart50_m2m    False
dtype: bool


In [19]:
print(df.isnull().sum().sum())

0


In [20]:
from sklearn.preprocessing import MinMaxScaler
original_values = df['score'].values.reshape(-1, 1)
scaler = MinMaxScaler(feature_range=(0, 1), copy=True)
normalized_values = scaler.fit_transform(original_values)
df['normalized_score'] = normalized_values
original_values_restored = scaler.inverse_transform(normalized_values)
df['restored_score'] = original_values_restored

In [21]:
df.head()

,Unnamed: 0,essay_id,essay_set,essay,score,essay_hindi,mbart50_m2m,normalized_score,restored_score
0,10686,17834,7,Patience is when your waiting .I was patience ...,15,सब्र तब होता है जब आप इंतजार कर रहे होते हैं। ...,धैर्य जब तपाईँले पर्खिरहनु भएको हो म लंचको लाग...,0.590909,15.0
1,10687,17836,7,"I am not a patience person, like I can’t sit i...",13,"मैं एक धैर्यवान व्यक्ति नहीं हूं, जैसे कि मैं ...",म चिन्तित व्यक्ति होइन जस्तो म पाँच मिनेटभन्दा...,0.500000,13.0
2,10688,17837,7,One day I was at basketball practice and I was...,15,एक दिन मैं बास्केटबॉल अभ्यास में था और मैं अपन...,एक दिन म बास्केटबाल अभ्यासमा थिएँ र मैले आफ्नो...,0.590909,15.0
3,10689,17838,7,I going to write about a time when I went to t...,17,मैं उस समय के बारे में लिखने जा रहा हूं जब मैं...,फेयरमा जाँदा हामीले रमाइलो गर्यौं हामीले हेर्न...,0.681818,17.0
4,10690,17839,7,It can be very hard for somebody to be patient...,13,किसी के लिए धैर्य रखना बहुत कठिन हो सकता है। य...,कसैको लागि धैर्य धेरै कठिन हुन सक्छ यदि तिमी ध...,0.500000,13.0


In [22]:
df.mbart50_m2m

0       धैर्य जब तपाईँले पर्खिरहनु भएको हो म लंचको लाग...
1       म चिन्तित व्यक्ति होइन जस्तो म पाँच मिनेटभन्दा...
2       एक दिन म बास्केटबाल अभ्यासमा थिएँ र मैले आफ्नो...
3       फेयरमा जाँदा हामीले रमाइलो गर्यौं हामीले हेर्न...
4       कसैको लागि धैर्य धेरै कठिन हुन सक्छ यदि तिमी ध...
                              ...                        
1564    एक पटक मैले एउटा राम्रो खेल प्राप्त गरिरहेको थ...
1565    मेरो जीवनमा एक पेटेन्ट व्यक्ति मेरी आमा हुन् अ...
1566    मेरो आमाले काम खोजिरहेको बेला मलाई थाहा भएको अ...
1567    म विवाहलाई घृणा गर्दछु म मानिसहरूलाई विवाह गर्...
1568    केही हप्ता अघि हामीले गैराजको बिक्री गर्यौं र ...
Name: mbart50_m2m, Length: 1569, dtype: object

In [23]:
highest_score = df['score'].max()
print(highest_score)

24


In [24]:
highest_score = df['normalized_score'].max()
print(highest_score)

0.9999999999999999


In [25]:
df['score'].value_counts()

16    199
17    160
18    118
14    105
20     99
24     96
19     88
12     86
15     85
13     82
21     68
22     62
11     56
10     55
23     53
8      50
9      49
7      28
6      20
4       4
5       4
2       1
3       1
Name: score, dtype: int64

In [26]:
df['normalized_score'].value_counts()

0.636364    199
0.681818    160
0.727273    118
0.545455    105
0.818182     99
1.000000     96
0.772727     88
0.454545     86
0.590909     85
0.500000     82
0.863636     68
0.909091     62
0.409091     56
0.363636     55
0.954545     53
0.272727     50
0.318182     49
0.227273     28
0.181818     20
0.090909      4
0.136364      4
0.000000      1
0.045455      1
Name: normalized_score, dtype: int64

In [27]:
df.head()

,Unnamed: 0,essay_id,essay_set,essay,score,essay_hindi,mbart50_m2m,normalized_score,restored_score
0,10686,17834,7,Patience is when your waiting .I was patience ...,15,सब्र तब होता है जब आप इंतजार कर रहे होते हैं। ...,धैर्य जब तपाईँले पर्खिरहनु भएको हो म लंचको लाग...,0.590909,15.0
1,10687,17836,7,"I am not a patience person, like I can’t sit i...",13,"मैं एक धैर्यवान व्यक्ति नहीं हूं, जैसे कि मैं ...",म चिन्तित व्यक्ति होइन जस्तो म पाँच मिनेटभन्दा...,0.500000,13.0
2,10688,17837,7,One day I was at basketball practice and I was...,15,एक दिन मैं बास्केटबॉल अभ्यास में था और मैं अपन...,एक दिन म बास्केटबाल अभ्यासमा थिएँ र मैले आफ्नो...,0.590909,15.0
3,10689,17838,7,I going to write about a time when I went to t...,17,मैं उस समय के बारे में लिखने जा रहा हूं जब मैं...,फेयरमा जाँदा हामीले रमाइलो गर्यौं हामीले हेर्न...,0.681818,17.0
4,10690,17839,7,It can be very hard for somebody to be patient...,13,किसी के लिए धैर्य रखना बहुत कठिन हो सकता है। य...,कसैको लागि धैर्य धेरै कठिन हुन सक्छ यदि तिमी ध...,0.500000,13.0


### amitness/roberta-base-ne

In [ ]:
# Custom Dataset
class EssayDataset(Dataset):
    def __init__(self, essays, scores, tokenizer, max_len):
        self.essays = essays
        self.scores = scores
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.essays)

    def __getitem__(self, item):
        essay = str(self.essays[item])
        score = self.scores[item]

        encoding = self.tokenizer.encode_plus(
            essay,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            return_attention_mask=True,
            return_tensors='pt',
            truncation=True
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'score': torch.tensor(score, dtype=torch.float)
        }

In [ ]:
# Model and Tokenizer
tokenizer = RobertaTokenizer.from_pretrained('amitness/roberta-base-ne')
model = RobertaForSequenceClassification.from_pretrained('amitness/roberta-base-ne', num_labels=1)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at amitness/roberta-base-ne and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# Splitting data
train_texts, temp_texts, train_scores, temp_scores = train_test_split(df['mbart50_m2m'], df['normalized_score'], test_size=0.3)

# Reset index
train_texts = train_texts.reset_index(drop=True)
temp_texts = temp_texts.reset_index(drop=True)
train_scores = train_scores.reset_index(drop=True)
temp_scores = temp_scores.reset_index(drop=True)

# val data and test data
val_texts, test_texts, val_scores, test_scores = train_test_split(temp_texts, temp_scores, test_size=0.5)

# Reset index
val_texts = val_texts.reset_index(drop=True)
test_texts = test_texts.reset_index(drop=True)
val_scores = val_scores.reset_index(drop=True)
test_scores = test_scores.reset_index(drop=True)


train_dataset = EssayDataset(train_texts, train_scores, tokenizer, max_len=512)
val_dataset = EssayDataset(val_texts, val_scores, tokenizer, max_len=512)
test_dataset = EssayDataset(test_texts, test_scores, tokenizer, max_len=512)

train_loader = DataLoader(train_dataset, batch_size=8)
val_loader = DataLoader(val_dataset, batch_size=8)
test_loader = DataLoader(test_dataset, batch_size=8)

In [ ]:
# Training
optimizer = AdamW(model.parameters(), lr=2e-5)
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    total_train_loss = 0
    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        scores = batch['score'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits.squeeze()
        loss = torch.nn.functional.mse_loss(logits, scores)
        total_train_loss += loss.item()
        loss.backward()
        optimizer.step()
    avg_train_loss = total_train_loss / len(train_loader)

    # Validation
    model.eval()
    total_val_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            scores = batch['score'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits.squeeze()
            loss = torch.nn.functional.mse_loss(logits, scores)
            total_val_loss += loss.item()
    avg_val_loss = total_val_loss / len(val_loader)

    print(f'Epoch {epoch + 1}/{num_epochs} | Train Loss: {avg_train_loss} | Val Loss: {avg_val_loss}')

/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:429: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Epoch 1/10 | Train Loss: 0.05012063139482685 | Val Loss: 0.033840250596404074
Epoch 2/10 | Train Loss: 0.036796171882905175 | Val Loss: 0.02376370389635364
Epoch 3/10 | Train Loss: 0.03446866415333057 | Val Loss: 0.045137576883037885
Epoch 4/10 | Train Loss: 0.031713108896561294 | Val Loss: 0.024886633036658168
Epoch 5/10 | Train Loss: 0.02870500729540768 | Val Loss: 0.02113733923373123
Epoch 6/10 | Train Loss: 0.027236219286324755 | Val Loss: 0.0364516602208217
Epoch 7/10 | Train Loss: 0.02451244846958181 | Val Loss: 0.022179127329339583
Epoch 8/10 | Train Loss: 0.022894295782822628 | Val Loss: 0.0242174136141936
Epoch 9/10 | Train Loss: 0.02601510578887942 | Val Loss: 0.0302486896670113
Epoch 10/10 | Train Loss: 0.032401109357242996 | Val Loss: 0.025126619706861676


In [ ]:
# Evaluation
def evaluate_model(model, test_loader):
    model.eval()
    predictions, actuals = [], []
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            scores = batch['score'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            predictions.extend(outputs.logits.squeeze().cpu().numpy())
            actuals.extend(scores.cpu().numpy())

    # Convert to arrays
    predictions = np.array(predictions)
    actuals = np.array(actuals)

    predictions_inv_tras = (scaler.inverse_transform(predictions.reshape(-1, 1)).squeeze())
    actuals_inv_tras = (scaler.inverse_transform(actuals.reshape(-1, 1)).squeeze())

    mse = mean_squared_error(actuals, predictions)
    mae = mean_absolute_error(actuals, predictions)
    r2 = r2_score(actuals, predictions)
    pearson_corr = pearsonr(actuals, predictions)[0]
    spearman_corr = spearmanr(actuals, predictions)[0]
    qwk = cohen_kappa_score(np.round(actuals_inv_tras), np.round(predictions_inv_tras), weights='quadratic')

    return mse, mae, r2, pearson_corr, spearman_corr, qwk

mse, mae, r2, pearson_corr, spearman_corr, qwk = evaluate_model(model, test_loader)
print(f'MSE: {mse}')
print(f'MAE: {mae}')
print(f'R2: {r2}')
print(f'Pearson Correlation: {pearson_corr}')
print(f'Spearman Correlation: {spearman_corr}')
print(f'QWK: {qwk}')

MSE: 0.021574635058641434
MAE: 0.11382728070020676
R2: 0.5105014517217794
Pearson Correlation: 0.7467543831765056
Spearman Correlation: 0.7312899001243427
QWK: 0.7451321727917473




MSE: 0.01683727838099003
MAE: 0.09970073401927948
R2: 0.36088829195233807
Pearson Correlation: 0.6541778233775262
Spearman Correlation: 0.5376987551112392
QWK: 0.6119408735407811

### NepBERTa/NepBERTa

In [ ]:
# Custom Dataset
class EssayDataset(Dataset):
    def __init__(self, essays, scores, tokenizer, max_len):
        self.essays = essays
        self.scores = scores
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.essays)

    def __getitem__(self, item):
        essay = str(self.essays[item])
        score = self.scores[item]

        encoding = self.tokenizer.encode_plus(
            essay,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            return_attention_mask=True,
            return_tensors='pt',
            truncation=True
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'score': torch.tensor(score, dtype=torch.float)
        }


tokenizer = BertTokenizer.from_pretrained('NepBERTa/NepBERTa')
model = BertForSequenceClassification.from_pretrained('NepBERTa/NepBERTa', num_labels=1, from_tf=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Splitting data
train_texts, temp_texts, train_scores, temp_scores = train_test_split(df['mbart50_m2m'], df['normalized_score'], test_size=0.3)

# Reset index
train_texts = train_texts.reset_index(drop=True)
temp_texts = temp_texts.reset_index(drop=True)
train_scores = train_scores.reset_index(drop=True)
temp_scores = temp_scores.reset_index(drop=True)

# val data and test data
val_texts, test_texts, val_scores, test_scores = train_test_split(temp_texts, temp_scores, test_size=0.5)

# Reset index
val_texts = val_texts.reset_index(drop=True)
test_texts = test_texts.reset_index(drop=True)
val_scores = val_scores.reset_index(drop=True)
test_scores = test_scores.reset_index(drop=True)


train_dataset = EssayDataset(train_texts, train_scores, tokenizer, max_len=512)
val_dataset = EssayDataset(val_texts, val_scores, tokenizer, max_len=512)
test_dataset = EssayDataset(test_texts, test_scores, tokenizer, max_len=512)

train_loader = DataLoader(train_dataset, batch_size=8)
val_loader = DataLoader(val_dataset, batch_size=8)
test_loader = DataLoader(test_dataset, batch_size=8)

# Training
optimizer = AdamW(model.parameters(), lr=2e-5)
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    total_train_loss = 0
    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        scores = batch['score'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits.squeeze()
        loss = torch.nn.functional.mse_loss(logits, scores)
        total_train_loss += loss.item()
        loss.backward()
        optimizer.step()
    avg_train_loss = total_train_loss / len(train_loader)

    # Validation
    model.eval()
    total_val_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            scores = batch['score'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits.squeeze()
            loss = torch.nn.functional.mse_loss(logits, scores)
            total_val_loss += loss.item()
    avg_val_loss = total_val_loss / len(val_loader)

    print(f'Epoch {epoch + 1}/{num_epochs} | Train Loss: {avg_train_loss} | Val Loss: {avg_val_loss}')

# max_score = df['score'].max()
# Evaluation
def evaluate_model(model, test_loader):
    model.eval()
    predictions, actuals = [], []
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            scores = batch['score'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            predictions.extend(outputs.logits.squeeze().cpu().numpy())
            actuals.extend(scores.cpu().numpy())

    predictions = np.array(predictions)
    actuals = np.array(actuals)

    # print("Predictions: ", predictions)
    # print("Actuals: ", actuals)

    predictions_inv_tras = (scaler.inverse_transform(predictions.reshape(-1, 1)).squeeze())
    actuals_inv_tras = (scaler.inverse_transform(actuals.reshape(-1, 1)).squeeze())
    # print("Predictions_inv_tras: ", predictions_inv_tras)
    # print("Actuals_inv_tras: ", actuals_inv_tras)


    mse = mean_squared_error(actuals, predictions)
    mae = mean_absolute_error(actuals, predictions)
    r2 = r2_score(actuals, predictions)
    pearson_corr = pearsonr(actuals, predictions)[0]
    spearman_corr = spearmanr(actuals, predictions)[0]
    qwk = cohen_kappa_score(np.round(actuals_inv_tras), np.round(predictions_inv_tras), weights='quadratic')

    return mse, mae, r2, pearson_corr, spearman_corr, qwk

mse, mae, r2, pearson_corr, spearman_corr, qwk = evaluate_model(model, test_loader)
print(f'MSE: {mse}')
print(f'MAE: {mae}')
print(f'R2: {r2}')
print(f'Pearson Correlation: {pearson_corr}')
print(f'Spearman Correlation: {spearman_corr}')
print(f'QWK: {qwk}')

All TF 2.0 model weights were used when initializing BertForSequenceClassification.

All the weights of BertForSequenceClassification were initialized from the TF 2.0 model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use BertForSequenceClassification for predictions without further training.
/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:429: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Epoch 1/10 | Train Loss: 0.0303815881708178 | Val Loss: 0.02134162443301951
Epoch 2/10 | Train Loss: 0.020426817837595583 | Val Loss: 0.02307441458106041
Epoch 3/10 | Train Loss: 0.016327249318145325 | Val Loss: 0.022653721055636802
Epoch 4/10 | Train Loss: 0.01354063004238423 | Val Loss: 0.01742210236067573
Epoch 5/10 | Train Loss: 0.011499340385299824 | Val Loss: 0.01964445683018615
Epoch 6/10 | Train Loss: 0.00930760125460886 | Val Loss: 0.017550086804355183
Epoch 7/10 | Train Loss: 0.011131865433012337 | Val Loss: 0.019057328548903265
Epoch 8/10 | Train Loss: 0.011240010598566438 | Val Loss: 0.01852254355326295
Epoch 9/10 | Train Loss: 0.009849234058123513 | Val Loss: 0.029879104144250355
Epoch 10/10 | Train Loss: 0.011086186290333939 | Val Loss: 0.029639112064614892
MSE: 0.034141335636377335
MAE: 0.1466425657272339
R2: 0.21948912291132272
Pearson Correlation: 0.7464289127439752
Spearman Correlation: 0.7374878841986405
QWK: 0.6939274685395923


MSE: 0.014094462618231773
MAE: 0.0913693904876709
R2: 0.4616231489504393
Pearson Correlation: 0.8312037298894959
Spearman Correlation: 0.7699140596696659
QWK: 0.6625091953401965

In [ ]:
print("Train dataset: ", len(train_dataset))
print("Val dataset: ", len(val_dataset))
print("Val dataset: ", len(test_dataset))

Train dataset:  1098
Val dataset:  235
Val dataset:  236


### Shushant/NepNewsBERT

In [28]:
# Custom Dataset
class EssayDataset(Dataset):
    def __init__(self, essays, scores, tokenizer, max_len):
        self.essays = essays
        self.scores = scores
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.essays)

    def __getitem__(self, item):
        essay = str(self.essays[item])
        score = self.scores[item]

        encoding = self.tokenizer.encode_plus(
            essay,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            return_attention_mask=True,
            return_tensors='pt',
            truncation=True
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'score': torch.tensor(score, dtype=torch.float)
        }

tokenizer = BertTokenizer.from_pretrained('Shushant/NepNewsBERT')
model = BertForSequenceClassification.from_pretrained('Shushant/NepNewsBERT', num_labels=1)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Splitting data
train_texts, temp_texts, train_scores, temp_scores = train_test_split(df['mbart50_m2m'], df['normalized_score'], test_size=0.3)

# Reset index
train_texts = train_texts.reset_index(drop=True)
temp_texts = temp_texts.reset_index(drop=True)
train_scores = train_scores.reset_index(drop=True)
temp_scores = temp_scores.reset_index(drop=True)

# val data and test data
val_texts, test_texts, val_scores, test_scores = train_test_split(temp_texts, temp_scores, test_size=0.5)

# Reset index
val_texts = val_texts.reset_index(drop=True)
test_texts = test_texts.reset_index(drop=True)
val_scores = val_scores.reset_index(drop=True)
test_scores = test_scores.reset_index(drop=True)


train_dataset = EssayDataset(train_texts, train_scores, tokenizer, max_len=512)
val_dataset = EssayDataset(val_texts, val_scores, tokenizer, max_len=512)
test_dataset = EssayDataset(test_texts, test_scores, tokenizer, max_len=512)

train_loader = DataLoader(train_dataset, batch_size=8)
val_loader = DataLoader(val_dataset, batch_size=8)
test_loader = DataLoader(test_dataset, batch_size=8)

# Training
optimizer = AdamW(model.parameters(), lr=2e-5)
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    total_train_loss = 0
    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        scores = batch['score'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits.squeeze()
        loss = torch.nn.functional.mse_loss(logits, scores)
        total_train_loss += loss.item()
        loss.backward()
        optimizer.step()
    avg_train_loss = total_train_loss / len(train_loader)

    # Validation
    model.eval()
    total_val_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            scores = batch['score'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits.squeeze()
            loss = torch.nn.functional.mse_loss(logits, scores)
            total_val_loss += loss.item()
    avg_val_loss = total_val_loss / len(val_loader)

    print(f'Epoch {epoch + 1}/{num_epochs} | Train Loss: {avg_train_loss} | Val Loss: {avg_val_loss}')

# max_score = df['score'].max()
# Evaluation
def evaluate_model(model, test_loader):
    model.eval()
    predictions, actuals = [], []
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            scores = batch['score'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            predictions.extend(outputs.logits.squeeze().cpu().numpy())
            actuals.extend(scores.cpu().numpy())

    predictions = np.array(predictions)
    actuals = np.array(actuals)
    # print("Predictions: ", predictions)
    # print("Actuals: ", actuals)

    predictions_inv_tras = (scaler.inverse_transform(predictions.reshape(-1, 1)).squeeze())
    actuals_inv_tras = (scaler.inverse_transform(actuals.reshape(-1, 1)).squeeze())
    # print("Predictions_inv_tras: ", predictions_inv_tras)
    # print("Actuals_inv_tras: ", actuals_inv_tras)


    mse = mean_squared_error(actuals, predictions)
    mae = mean_absolute_error(actuals, predictions)
    r2 = r2_score(actuals, predictions)
    pearson_corr = pearsonr(actuals, predictions)[0]
    spearman_corr = spearmanr(actuals, predictions)[0]
    qwk = cohen_kappa_score(np.round(actuals_inv_tras), np.round(predictions_inv_tras), weights='quadratic')

    return mse, mae, r2, pearson_corr, spearman_corr, qwk

mse, mae, r2, pearson_corr, spearman_corr, qwk = evaluate_model(model, test_loader)
print(f'MSE: {mse}')
print(f'MAE: {mae}')
print(f'R2: {r2}')
print(f'Pearson Correlation: {pearson_corr}')
print(f'Spearman Correlation: {spearman_corr}')
print(f'QWK: {qwk}')

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


vocab.txt:   0%|          | 0.00/600k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/589 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at Shushant/NepNewsBERT and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:429: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warn

Epoch 1/10 | Train Loss: 0.05744045939417961 | Val Loss: 0.05639784342298905
Epoch 2/10 | Train Loss: 0.03519588624181199 | Val Loss: 0.05536726471036672
Epoch 3/10 | Train Loss: 0.03689971043174897 | Val Loss: 0.026525353205700715
Epoch 4/10 | Train Loss: 0.027460628236387518 | Val Loss: 0.018446666731809575
Epoch 5/10 | Train Loss: 0.022934099335385406 | Val Loss: 0.017836637375876308
Epoch 6/10 | Train Loss: 0.02125205622608031 | Val Loss: 0.02610389421073099
Epoch 7/10 | Train Loss: 0.02316321248350584 | Val Loss: 0.025685166691740355
Epoch 8/10 | Train Loss: 0.021897639257603907 | Val Loss: 0.04550495098034541
Epoch 9/10 | Train Loss: 0.020229830349242126 | Val Loss: 0.05483307397613923
Epoch 10/10 | Train Loss: 0.020390517316232232 | Val Loss: 0.023419689169774452
MSE: 0.02579839713871479
MAE: 0.1288565993309021
R2: 0.39019415812680347
Pearson Correlation: 0.7927612467944067
Spearman Correlation: 0.7975968468724525
QWK: 0.7081360793103564


MSE: 0.023322373628616333
MAE: 0.12150432169437408
R2: 0.6277407562705817
Pearson Correlation: 0.8267204079669765
Spearman Correlation: 0.7986475027740367
QWK: 0.7519221439807124

### Rajan/NepaliBERT

In [29]:
# Custom Dataset
class EssayDataset(Dataset):
    def __init__(self, essays, scores, tokenizer, max_len):
        self.essays = essays
        self.scores = scores
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.essays)

    def __getitem__(self, item):
        essay = str(self.essays[item])
        score = self.scores[item]

        encoding = self.tokenizer.encode_plus(
            essay,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            return_attention_mask=True,
            return_tensors='pt',
            truncation=True
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'score': torch.tensor(score, dtype=torch.float)
        }


tokenizer = BertTokenizer.from_pretrained('Rajan/NepaliBERT')
model = BertForSequenceClassification.from_pretrained('Rajan/NepaliBERT', num_labels=1)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Splitting data
train_texts, temp_texts, train_scores, temp_scores = train_test_split(df['mbart50_m2m'], df['normalized_score'], test_size=0.3)

# Reset index
train_texts = train_texts.reset_index(drop=True)
temp_texts = temp_texts.reset_index(drop=True)
train_scores = train_scores.reset_index(drop=True)
temp_scores = temp_scores.reset_index(drop=True)

# val data and test data
val_texts, test_texts, val_scores, test_scores = train_test_split(temp_texts, temp_scores, test_size=0.5)

# Reset index
val_texts = val_texts.reset_index(drop=True)
test_texts = test_texts.reset_index(drop=True)
val_scores = val_scores.reset_index(drop=True)
test_scores = test_scores.reset_index(drop=True)


train_dataset = EssayDataset(train_texts, train_scores, tokenizer, max_len=512)
val_dataset = EssayDataset(val_texts, val_scores, tokenizer, max_len=512)
test_dataset = EssayDataset(test_texts, test_scores, tokenizer, max_len=512)

train_loader = DataLoader(train_dataset, batch_size=8)
val_loader = DataLoader(val_dataset, batch_size=8)
test_loader = DataLoader(test_dataset, batch_size=8)

# Training
optimizer = AdamW(model.parameters(), lr=2e-5)
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    total_train_loss = 0
    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        scores = batch['score'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits.squeeze()
        loss = torch.nn.functional.mse_loss(logits, scores)
        total_train_loss += loss.item()
        loss.backward()
        optimizer.step()
    avg_train_loss = total_train_loss / len(train_loader)

    # Validation
    model.eval()
    total_val_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            scores = batch['score'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits.squeeze()
            loss = torch.nn.functional.mse_loss(logits, scores)
            total_val_loss += loss.item()
    avg_val_loss = total_val_loss / len(val_loader)

    print(f'Epoch {epoch + 1}/{num_epochs} | Train Loss: {avg_train_loss} | Val Loss: {avg_val_loss}')

# max_score = df['score'].max()
# Evaluation
def evaluate_model(model, test_loader):
    model.eval()
    predictions, actuals = [], []
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            scores = batch['score'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            predictions.extend(outputs.logits.squeeze().cpu().numpy())
            actuals.extend(scores.cpu().numpy())

    predictions = np.array(predictions)
    actuals = np.array(actuals)
    # print("Predictions: ", predictions)
    # print("Actuals: ", actuals)

    predictions_inv_tras = (scaler.inverse_transform(predictions.reshape(-1, 1)).squeeze())
    actuals_inv_tras = (scaler.inverse_transform(actuals.reshape(-1, 1)).squeeze())
    # print("Predictions_inv_tras: ", predictions_inv_tras)
    # print("Actuals_inv_tras: ", actuals_inv_tras)


    mse = mean_squared_error(actuals, predictions)
    mae = mean_absolute_error(actuals, predictions)
    r2 = r2_score(actuals, predictions)
    pearson_corr = pearsonr(actuals, predictions)[0]
    spearman_corr = spearmanr(actuals, predictions)[0]
    qwk = cohen_kappa_score(np.round(actuals_inv_tras), np.round(predictions_inv_tras), weights='quadratic')

    return mse, mae, r2, pearson_corr, spearman_corr, qwk

mse, mae, r2, pearson_corr, spearman_corr, qwk = evaluate_model(model, test_loader)
print(f'MSE: {mse}')
print(f'MAE: {mae}')
print(f'R2: {r2}')
print(f'Pearson Correlation: {pearson_corr}')
print(f'Spearman Correlation: {spearman_corr}')
print(f'QWK: {qwk}')

vocab.txt:   0%|          | 0.00/987k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/569 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/328M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at Rajan/NepaliBERT and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:429: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Epoch 1/10 | Train Loss: 0.04992321035757229 | Val Loss: 0.04366442255365352
Epoch 2/10 | Train Loss: 0.03033799055855775 | Val Loss: 0.030442841785649457
Epoch 3/10 | Train Loss: 0.029538458270574178 | Val Loss: 0.028164799379495283
Epoch 4/10 | Train Loss: 0.02611889881803148 | Val Loss: 0.022339209665854773
Epoch 5/10 | Train Loss: 0.02382549631368855 | Val Loss: 0.02871421022961537
Epoch 6/10 | Train Loss: 0.020289622699382944 | Val Loss: 0.02338669206947088
Epoch 7/10 | Train Loss: 0.018158336423094504 | Val Loss: 0.04973512062182029
Epoch 8/10 | Train Loss: 0.021147905796617808 | Val Loss: 0.030321799761926133
Epoch 9/10 | Train Loss: 0.01841871703610472 | Val Loss: 0.027787783555686472
Epoch 10/10 | Train Loss: 0.01782970763641693 | Val Loss: 0.02391501938303312
MSE: 0.021394839510321617
MAE: 0.11138433963060379
R2: 0.4935081116483302
Pearson Correlation: 0.7512840733234396
Spearman Correlation: 0.745008323259106
QWK: 0.7451180412608316


MSE: 0.030582893639802933
MAE: 0.142673522233963
R2: 0.45224659380275034
Pearson Correlation: 0.8173120915146391
Spearman Correlation: 0.7584970714559526
QWK: 0.7212869435091658

### dexhrestha/Nepali-DistilBERT

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("dexhrestha/Nepali-DistilBERT")
max_length = tokenizer.model_max_length
print("Max sequence length:", max_length)

In [30]:
# seed = 128
# torch.manual_seed(seed)
# torch.cuda.manual_seed(seed)
# torch.cuda.manual_seed_all(seed)

# if torch.cuda.is_available():
#     torch.backends.cudnn.deterministic = True
#     torch.backends.cudnn.benchmark = False

# np.random.seed(seed)

# Custom Dataset
class EssayDataset(Dataset):
    def __init__(self, essays, scores, tokenizer, max_len):
        self.essays = essays
        self.scores = scores
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.essays)

    def __getitem__(self, item):
        essay = str(self.essays[item])
        score = self.scores[item]

        encoding = self.tokenizer.encode_plus(
            essay,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            return_attention_mask=True,
            return_tensors='pt',
            truncation=True
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'score': torch.tensor(score, dtype=torch.float)
        }


tokenizer = AutoTokenizer.from_pretrained('dexhrestha/Nepali-DistilBERT')
model = AutoModelForSequenceClassification.from_pretrained('dexhrestha/Nepali-DistilBERT', num_labels=1, ignore_mismatched_sizes=True)
# model = DistilBertForSequenceClassification.from_pretrained('dexhrestha/Nepali-DistilBERT', num_labels=1, ignore_mismatched_sizes=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Splitting data
train_texts, temp_texts, train_scores, temp_scores = train_test_split(df['mbart50_m2m'], df['normalized_score'], test_size=0.3)

# Reset index
train_texts = train_texts.reset_index(drop=True)
temp_texts = temp_texts.reset_index(drop=True)
train_scores = train_scores.reset_index(drop=True)
temp_scores = temp_scores.reset_index(drop=True)

# val data and test data
val_texts, test_texts, val_scores, test_scores = train_test_split(temp_texts, temp_scores, test_size=0.5)

# Reset index
val_texts = val_texts.reset_index(drop=True)
test_texts = test_texts.reset_index(drop=True)
val_scores = val_scores.reset_index(drop=True)
test_scores = test_scores.reset_index(drop=True)

max_len = 128
train_dataset = EssayDataset(train_texts, train_scores, tokenizer, max_len=max_len)
val_dataset = EssayDataset(val_texts, val_scores, tokenizer, max_len=max_len)
test_dataset = EssayDataset(test_texts, test_scores, tokenizer, max_len=max_len)

train_loader = DataLoader(train_dataset, batch_size=8)
val_loader = DataLoader(val_dataset, batch_size=8)
test_loader = DataLoader(test_dataset, batch_size=8)

# Training
optimizer = AdamW(model.parameters(), lr=2e-5)
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    total_train_loss = 0
    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        scores = batch['score'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits.squeeze()
        loss = torch.nn.functional.mse_loss(logits, scores)
        total_train_loss += loss.item()
        loss.backward()
        optimizer.step()
    avg_train_loss = total_train_loss / len(train_loader)

    # Validation
    model.eval()
    total_val_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            scores = batch['score'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits.squeeze()
            loss = torch.nn.functional.mse_loss(logits, scores)
            total_val_loss += loss.item()
    avg_val_loss = total_val_loss / len(val_loader)

    print(f'Epoch {epoch + 1}/{num_epochs} | Train Loss: {avg_train_loss} | Val Loss: {avg_val_loss}')

# max_score = df['score'].max()
# Evaluation
def evaluate_model(model, test_loader):
    model.eval()
    predictions, actuals = [], []
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            scores = batch['score'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            predictions.extend(outputs.logits.squeeze().cpu().numpy())
            actuals.extend(scores.cpu().numpy())

    predictions = np.array(predictions)
    actuals = np.array(actuals)
    # print("Predictions: ", predictions)
    # print("Actuals: ", actuals)

    predictions_inv_tras = (scaler.inverse_transform(predictions.reshape(-1, 1)).squeeze())
    actuals_inv_tras = (scaler.inverse_transform(actuals.reshape(-1, 1)).squeeze())
    # print("Predictions_inv_tras: ", predictions_inv_tras)
    # print("Actuals_inv_tras: ", actuals_inv_tras)


    mse = mean_squared_error(actuals, predictions)
    mae = mean_absolute_error(actuals, predictions)
    r2 = r2_score(actuals, predictions)
    pearson_corr = pearsonr(actuals, predictions)[0]
    spearman_corr = spearmanr(actuals, predictions)[0]
    qwk = cohen_kappa_score(np.round(actuals_inv_tras), np.round(predictions_inv_tras), weights='quadratic')

    return mse, mae, r2, pearson_corr, spearman_corr, qwk

mse, mae, r2, pearson_corr, spearman_corr, qwk = evaluate_model(model, test_loader)
print(f'MSE: {mse}')
print(f'MAE: {mae}')
print(f'R2: {r2}')
print(f'Pearson Correlation: {pearson_corr}')
print(f'Spearman Correlation: {spearman_corr}')
print(f'QWK: {qwk}')

tokenizer_config.json:   0%|          | 0.00/399 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/489k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/723k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/648 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/267M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at dexhrestha/Nepali-DistilBERT and are newly initialized because the shapes did not match:
- classifier.weight: found shape torch.Size([2, 768]) in the checkpoint and torch.Size([1, 768]) in the model instantiated
- classifier.bias: found shape torch.Size([2]) in the checkpoint and torch.Size([1]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:429: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Epoch 1/10 | Train Loss: 0.051123642544869494 | Val Loss: 0.01935983539248506
Epoch 2/10 | Train Loss: 0.027674038555351613 | Val Loss: 0.019196086563169956
Epoch 3/10 | Train Loss: 0.02057051584521052 | Val Loss: 0.01814053622074425
Epoch 4/10 | Train Loss: 0.019105972830152165 | Val Loss: 0.021666699027021726
Epoch 5/10 | Train Loss: 0.013357372045942137 | Val Loss: 0.01933973780833185
Epoch 6/10 | Train Loss: 0.012286134388135828 | Val Loss: 0.019464621615285674
Epoch 7/10 | Train Loss: 0.014525353518800566 | Val Loss: 0.020307991948599616
Epoch 8/10 | Train Loss: 0.01655356559659476 | Val Loss: 0.03556150253862143
Epoch 9/10 | Train Loss: 0.013816673689069685 | Val Loss: 0.01847659917548299
Epoch 10/10 | Train Loss: 0.009086758464379101 | Val Loss: 0.017872590788950524
MSE: 0.019905589520931244
MAE: 0.11091747879981995
R2: 0.5685735186281986
Pearson Correlation: 0.7603634061126763
Spearman Correlation: 0.7426078350308573
QWK: 0.754831872728825


MSE: 0.02571851573884487
MAE: 0.13051073253154755
R2: 0.5414610332605101
Pearson Correlation: 0.7379458452283778
Spearman Correlation: 0.7034794517251182
QWK: 0.6564508156203659

In [ ]:
# # Set seed for reproducibility
# seed = 128
# torch.manual_seed(seed)
# torch.cuda.manual_seed(seed)
# torch.cuda.manual_seed_all(seed)

# if torch.cuda.is_available():
#     torch.backends.cudnn.deterministic = True
#     torch.backends.cudnn.benchmark = False

# np.random.seed(seed)

# # Custom Dataset
# class EssayDataset(Dataset):
#     def __init__(self, essays, scores, tokenizer, max_len):
#         self.essays = essays
#         self.scores = scores
#         self.tokenizer = tokenizer
#         self.max_len = max_len

#     def __len__(self):
#         return len(self.essays)

#     def __getitem__(self, item):
#         essay = str(self.essays[item])
#         score = self.scores[item]

#         encoding = self.tokenizer.encode_plus(
#             essay,
#             add_special_tokens=True,
#             max_length=self.max_len,
#             padding='max_length',
#             return_attention_mask=True,
#             return_tensors='pt',
#             truncation=True
#         )

#         return {
#             'input_ids': encoding['input_ids'].flatten(),
#             'attention_mask': encoding['attention_mask'].flatten(),
#             'score': torch.tensor(score, dtype=torch.float)
#         }

# # Splitting data
# train_texts, temp_texts, train_scores, temp_scores = train_test_split(df['mbart50_m2m'], df['normalized_score'], test_size=0.3)
# val_texts, test_texts, val_scores, test_scores = train_test_split(temp_texts, temp_scores, test_size=0.5)

# # Reset index
# train_texts, temp_texts, val_texts, test_texts = map(lambda x: x.reset_index(drop=True), [train_texts, temp_texts, val_texts, test_texts])
# train_scores, temp_scores, val_scores, test_scores = map(lambda x: x.reset_index(drop=True), [train_scores, temp_scores, val_scores, test_scores])

# # Model and Tokenizer
# tokenizer = AutoTokenizer.from_pretrained('dexhrestha/Nepali-DistilBERT')
# model = AutoModelForSequenceClassification.from_pretrained('dexhrestha/Nepali-DistilBERT', num_labels=1, ignore_mismatched_sizes=True)
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model = model.to(device)

# max_len = 128
# train_dataset = EssayDataset(train_texts, train_scores, tokenizer, max_len=max_len)
# val_dataset = EssayDataset(val_texts, val_scores, tokenizer, max_len=max_len)
# test_dataset = EssayDataset(test_texts, test_scores, tokenizer, max_len=max_len)

# # Training
# def train_model(trial, model, train_loader, val_loader):
#   try:
#       lr = trial.suggest_loguniform('lr', 1e-6, 1e-3)
#       weight_decay = trial.suggest_loguniform('weight_decay', 1e-6, 1e-3)
#       batch_size = trial.suggest_categorical('batch_size', [4, 8, 16])
#       num_epochs = trial.suggest_int('num_epochs', 1, 3)

#       optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
#       train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
#       val_loader = DataLoader(val_dataset, batch_size=batch_size)

#       for epoch in range(num_epochs):
#           model.train()
#           total_train_loss = 0
#           for batch in train_loader:
#               input_ids = batch['input_ids'].to(device)
#               attention_mask = batch['attention_mask'].to(device)
#               scores = batch['score'].to(device)

#               optimizer.zero_grad()
#               outputs = model(input_ids=input_ids, attention_mask=attention_mask)
#               logits = outputs.logits.squeeze()
#               loss = torch.nn.functional.mse_loss(logits, scores)
#               total_train_loss += loss.item()
#               loss.backward()
#               optimizer.step()

#           avg_train_loss = total_train_loss / len(train_loader)

#           # Validation
#           model.eval()
#           total_val_loss = 0
#           with torch.no_grad():
#               for batch in val_loader:
#                   input_ids = batch['input_ids'].to(device)
#                   attention_mask = batch['attention_mask'].to(device)
#                   scores = batch['score'].to(device)

#                   outputs = model(input_ids=input_ids, attention_mask=attention_mask)
#                   logits = outputs.logits.squeeze()
#                   loss = torch.nn.functional.mse_loss(logits, scores)
#                   total_val_loss += loss.item()

#           avg_val_loss = total_val_loss / len(val_loader)

#           print(f'Epoch {epoch + 1}/{num_epochs} | Train Loss: {avg_train_loss} | Val Loss: {avg_val_loss}')
#           return avg_val_loss
#   except Exception as e:
#           print(f"An error occurred: {e}")
#           return float('inf')  # Return a large value in case of an error

# # Evaluation
# def evaluate_model(model, test_loader):
#     model.eval()
#     predictions, actuals = [], []
#     with torch.no_grad():
#         for batch in test_loader:
#             input_ids = batch['input_ids'].to(device)
#             attention_mask = batch['attention_mask'].to(device)
#             scores = batch['score'].to(device)

#             outputs = model(input_ids=input_ids, attention_mask=attention_mask)
#             predictions.extend(outputs.logits.squeeze().cpu().numpy())
#             actuals.extend(scores.cpu().numpy())

#     predictions = np.array(predictions)
#     actuals = np.array(actuals)
#     # print("Predictions: ", predictions)
#     # print("Actuals: ", actuals)

#     predictions_inv_tras = (scaler.inverse_transform(predictions.reshape(-1, 1)).squeeze())
#     actuals_inv_tras = (scaler.inverse_transform(actuals.reshape(-1, 1)).squeeze())
#     # print("Predictions_inv_tras: ", predictions_inv_tras)
#     # print("Actuals_inv_tras: ", actuals_inv_tras)

#     mse = mean_squared_error(actuals, predictions)
#     mae = mean_absolute_error(actuals, predictions)
#     r2 = r2_score(actuals, predictions)
#     pearson_corr = pearsonr(actuals, predictions)[0]
#     spearman_corr = spearmanr(actuals, predictions)[0]
#     qwk = cohen_kappa_score(np.round(actuals_inv_tras), np.round(predictions_inv_tras), weights='quadratic')

#     return mse, mae, r2, pearson_corr, spearman_corr, qwk

# # Optuna study and optimization
# study = optuna.create_study(direction='minimize')
# study.optimize(lambda trial: train_model(trial, model, train_loader, val_loader), n_trials=50)

# # Get best hyperparameters from the study
# best_lr = study.best_params['lr']
# best_weight_decay = study.best_params['weight_decay']
# best_batch_size = study.best_params['batch_size']
# best_num_epochs = study.best_params['num_epochs']

# print(f'Best LR: {best_lr}')
# print(f'Best Weight Decay: {best_weight_decay}')
# print(f'Best Batch Size: {best_batch_size}')
# print(f'Best Num Epochs: {best_num_epochs}')

# # Use the best hyperparameters to train your final model
# optimizer = AdamW(model.parameters(), lr=best_lr, weight_decay=best_weight_decay)
# train_loader = DataLoader(train_dataset, batch_size=best_batch_size, shuffle=True)
# val_loader = DataLoader(val_dataset, batch_size=best_batch_size)

# # Evaluate final model
# mse, mae, r2, pearson_corr, spearman_corr, qwk = evaluate_model(model, test_loader)
# print(f'MSE: {mse}')
# print(f'MAE: {mae}')
# print(f'R2: {r2}')
# print(f'Pearson Correlation: {pearson_corr}')
# print(f'Spearman Correlation: {spearman_corr}')
# print(f'QWK: {qwk}')